# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Dataset Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://mlcommons.org/croissant/) library.

### Dataset Source
The dataset is defined by a Croissant schema, accessible at:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

- **Citation**: Liu, Y., Duan, X., Yang, S., Zhang, Y. and Han, S. (2026). Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution. Frontiers.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

# Print dataset name and description
print(f"{metadata['name']}\n{metadata['description']}")

## 2. Data Overview

Review available record sets, their fields, and unique `@id`s following the Croissant schema.

This is essential for exploring the structure of the dataset and referencing entities by their `@id`.

In [ ]:
# Get and display record sets and field @ids
record_sets = dataset.record_sets

if not record_sets:
    print("No record sets found in this dataset. Please check the 'recordSet' field in the metadata, or inspect the dataset contents.")
else:
    for rs in record_sets:
        print(f"\nRecord Set: {rs['@id']}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            if isinstance(field, dict):
                field_id = field.get('@id', '(no @id)')
                print(f"  Field: {field_id}")
            else:
                print(f"  Field: {field}")

## 3. Data Extraction

Load all available record sets by their `@id` into DataFrames for further analysis.

Reference all Croissant entities (record sets and fields) by their full `@id`.

In [ ]:
# List available record set @ids for extraction
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

if not record_set_ids:
    print("No record sets found. The dataset may be a single-table dataset; trying auto-discovery of tabular file objects.")
    # List all FileObjects
    if hasattr(dataset, 'file_objects') and dataset.file_objects:
        for fobj in dataset.file_objects:
            print(f"FileObject: {fobj['@id']} (contentUrl: {fobj.get('contentUrl', '(no contentUrl)')})")
    else:
        print("No FileObjects found in the metadata.")
else:
    for rsid in record_set_ids:
        print(f"\nLoading Record Set: {rsid}")
        records = list(dataset.records(record_set=rsid))
        if records:
            df = pd.DataFrame(records)
            dataframes[rsid] = df
            print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
        else:
            print(f"No records found for {rsid}.")

### If record sets are not defined (as may be the case for this dataset), try to load tabular data via FileObjects.

In [ ]:
# Try to find and load tabular data files if no record sets were defined
if not dataframes:
    # Attempt to retrieve FileObjects with format CSV/TSV
    if hasattr(dataset, 'file_objects') and dataset.file_objects:
        for fobj in dataset.file_objects:
            fo_id = fobj['@id']
            f_url = fobj.get('contentUrl')
            fmt = fobj.get('encodingFormat', '').lower()
            if f_url and (('csv' in fmt) or ('tsv' in fmt) or (f_url.endswith('.csv')) or (f_url.endswith('.tsv'))):
                print(f"Attempting to load FileObject {fo_id} as DataFrame from: {f_url}")
                try:
                    separator = ',' if 'csv' in fmt or (f_url and f_url.endswith('.csv')) else '\t'
                    df = pd.read_csv(f_url, sep=separator)
                    dataframes[fo_id] = df
                    print(f"Loaded shape: {df.shape}")
                except Exception as ex:
                    print(f"Could not load {f_url}: {ex}")
    else:
        print("No file objects with a suitable encoding format found.")

## 4. Exploratory Data Analysis (EDA)

In this section, we demonstrate a few typical processing and analysis operations.

- Filtering records based on a numeric field (e.g., `Age`)
- Normalizing that numeric field
- Grouping data by a categorical field (e.g., `Sex` or `Anatomical location`)

> **Note:** For demonstration, we identify likely numeric fields such as `Age` and group fields such as `Sex` by examining the DataFrame columns. We reference them explicitly by their column (field) name or corresponding `@id`, if available.

In [ ]:
# Find a DataFrame with record data
df_key = None
for key, df in dataframes.items():
    if not df.empty:
        df_key = key
        break

if df_key is None:
    print("No non-empty DataFrame found to analyze.")
else:
    print(f"Analyzing DataFrame from record set or FileObject: {df_key}")
    df = dataframes[df_key]
    print(f"Available fields (columns):\n{df.columns.tolist()}")

In [ ]:
# Example: Use 'Age' (or the closest available numeric field) for EDA
import numpy as np

# Try to pick the best numeric field
numeric_field_candidates = [c for c in df.columns if 'age' in c.lower()]
if not numeric_field_candidates:
    # Fallback: look for int/float types or obvious numeric fields
    possible_numeric = [col for col in df.columns if df[col].dtype in [np.int64, np.float64]]
    if possible_numeric:
        numeric_field = possible_numeric[0]
    else:
        numeric_field = df.columns[0]  # fallback to the first field
else:
    numeric_field = numeric_field_candidates[0]
print(f"Using numeric field: {numeric_field}")

try:
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
except Exception:
    pass

# Set threshold (e.g., age > 50 for older cancer survivors)
threshold = 50
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} records")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to group by a categorical field, e.g. 'Sex', 'AnatomicalLocation', etc.
possible_groups = [c for c in df.columns if c.lower() in ['sex', 'gender', 'anatomical location', 'anatomical_location', 'site', 'group']]
if possible_groups:
    group_field = possible_groups[0]
    print(f"\nGrouping by: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(grouped_df)
else:
    print("No obvious categorical group field found for grouping.")

## 5. Visualization

Visualize the distribution of the chosen numeric field and relationship with a group, if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df_key is not None and not filtered_df.empty:
    # Histogram of field
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f'Distribution of {numeric_field} (>{threshold})')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If grouped by field, show boxplot
    if possible_groups:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.title(f'{numeric_field} by {group_field} (>{threshold})')
        plt.show()

## 6. Conclusion

In this notebook, we loaded and explored the FAIR² dataset using the `mlcroissant` library and Croissant schema. We reviewed available record sets and fields using their `@id`s, loaded data into pandas, performed basic filtering, normalization, grouping, and created simple plots. The dataset provides clinical and molecular characteristics for cancer survivors with second primary colorectal cancer, supporting analyses of demographic and pathological factors. Further domain-specific analyses are encouraged.